In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
import os
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from src.config import SupabaseConfig

In [3]:
from src.risk_engine.portfolio import align_weights, portfolio_returns, covariance_matrix
from src.risk_engine.var_cvar import historical_var_cvar, parametric_var_cvar
from src.risk_config import CONFIDENCE_LEVELS, ROLLING_WINDOW_DAYS


In [4]:
load_dotenv()
cfg = SupabaseConfig.from_env()
engine = create_engine(cfg.sqlalchemy_url())

with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).scalar())

1


In [5]:
query = """
SELECT
    dd.date_id AS date,
    da.ticker,
    fr.simple_return
FROM fact_returns fr
JOIN dim_date dd ON fr.date_id = dd.date_id
JOIN dim_asset da ON fr.asset_id = da.asset_id
ORDER BY dd.date_id, da.ticker;
"""

returns_long = pd.read_sql(query, engine)
returns_long.shape

(76118, 3)

In [6]:
returns_long.head()

,date,ticker,simple_return
0,2005-01-04,AAPL,0.010270
1,2005-01-04,CAT,-0.011045
2,2005-01-04,DIS,-0.010772
3,2005-01-04,F,-0.003399
4,2005-01-04,GS,-0.006479


In [7]:
returns_df = returns_long.pivot(index='date', columns='ticker', values='simple_return')
returns_df.head()

ticker,AAPL,CAT,DIS,F,GS,JNJ,JPM,MDT,MSFT,NEE,NEM,PG,WMT,XOM
date,,,,,,,,,,,,,,
2005-01-04,0.010270,-0.011045,-0.010772,-0.003399,-0.006479,-0.003180,-0.010307,-0.004653,0.003739,-0.012179,-0.018650,-0.012502,-0.002436,-0.006788
2005-01-05,0.008758,-0.019145,-0.005445,-0.015689,-0.004507,-0.000638,0.002083,-0.007432,-0.002235,-0.012877,0.002406,0.010459,0.001315,-0.005226
2005-01-06,0.000776,0.014748,0.001460,0.001386,0.013777,0.002872,0.005716,0.020513,-0.001121,0.016237,-0.001680,0.004903,0.014262,0.012729
2005-01-07,0.072812,-0.002138,-0.009840,0.013841,-0.004276,-0.003660,-0.008009,0.009045,-0.002991,-0.001912,-0.002404,0.010480,-0.001110,-0.006584
2005-01-10,-0.004187,-0.009209,0.004049,-0.006826,0.001908,0.007826,-0.003385,0.023108,0.004874,0.003420,0.003856,0.007869,-0.005001,0.003816


In [8]:
query2 = """
SELECT
    da.ticker,
    dpw.weight
FROM dim_asset da
JOIN dim_portfolio_weight dpw ON da.asset_id = dpw.asset_id
JOIN dim_portfolio dp ON dpw.portfolio_id = dp.portfolio_id;
"""
weights_raw = pd.read_sql(query2, engine)
weights_raw

,ticker,weight
0,AAPL,0.07143
1,MSFT,0.07143
2,JPM,0.07143
3,GS,0.07143
4,XOM,0.07143
5,JNJ,0.07143
6,MDT,0.07143
7,PG,0.07143
8,WMT,0.07143
9,CAT,0.07143


In [9]:
weights = weights_raw.set_index('ticker')['weight']
weights

ticker
AAPL    0.07143
MSFT    0.07143
JPM     0.07143
GS      0.07143
XOM     0.07143
JNJ     0.07143
MDT     0.07143
PG      0.07143
WMT     0.07143
CAT     0.07143
NEE     0.07143
DIS     0.07143
F       0.07143
NEM     0.07143
Name: weight, dtype: float64

In [10]:
weights.sum()

np.float64(1.00002)

In [11]:
set(weights.index) == set(returns_df.columns)

True

In [12]:
weights_aligned = align_weights(weights, returns_df)
weights_aligned


ticker
AAPL    0.07143
CAT     0.07143
DIS     0.07143
F       0.07143
GS      0.07143
JNJ     0.07143
JPM     0.07143
MDT     0.07143
MSFT    0.07143
NEE     0.07143
NEM     0.07143
PG      0.07143
WMT     0.07143
XOM     0.07143
Name: weight, dtype: float64

In [13]:
port_returns = portfolio_returns(returns_df, weights)
port_returns.head()


date
2005-01-04   -0.006313
2005-01-05   -0.003441
2005-01-06    0.007613
2005-01-07    0.004518
2005-01-10    0.002294
dtype: float64

In [14]:
port_returns.sort_values(ascending = False)

date
2008-10-13    0.130265
2020-03-24    0.112275
2008-10-28    0.098597
2020-03-13    0.084396
2025-04-09    0.080074
                ...   
2020-03-09   -0.079733
2008-09-29   -0.082872
2008-12-01   -0.085246
2020-03-16   -0.089185
2020-03-12   -0.093044
Length: 5437, dtype: float64

In [15]:
port_returns_250 = port_returns.tail(ROLLING_WINDOW_DAYS)
port_returns_250


date
2025-08-18    0.002252
2025-08-19   -0.001517
2025-08-20    0.005639
2025-08-21   -0.006712
2025-08-22    0.015156
                ...   
2026-08-10    0.007695
2026-08-11    0.001850
2026-08-12    0.000560
2026-08-13    0.000217
2026-08-14    0.006693
Length: 250, dtype: float64

In [16]:
returns_df_250 = returns_df.tail(ROLLING_WINDOW_DAYS)


## Historical Simulation VaR / CVaR

`historical_var_cvar` and `parametric_var_cvar` are imported from `var_cvar.py` (see the import cell near the top), which in turn imports `align_weights`, `portfolio_returns`, and `covariance_matrix` from `portfolio.py` rather than reimplementing them — `portfolio.py` stays the single source of truth for weight alignment and return/covariance logic. Nothing is redefined locally in this notebook anymore, so what you validate here is exactly what `run_risk_pipeline.py` will call. Confidence levels and the rolling window size come from `risk_config.py` (`CONFIDENCE_LEVELS`, `ROLLING_WINDOW_DAYS`) rather than being hardcoded, so if you ever want to change them, that's the one place to do it.

In [17]:
var_95_full, cvar_95_full = historical_var_cvar(returns_df, weights, CONFIDENCE_LEVELS[0])
var_99_full, cvar_99_full = historical_var_cvar(returns_df, weights, CONFIDENCE_LEVELS[1])

print(f"Full history {CONFIDENCE_LEVELS[0]:.0%} VaR: {var_95_full:.4%}, CVaR: {cvar_95_full:.4%}")
print(f"Full history {CONFIDENCE_LEVELS[1]:.0%} VaR: {var_99_full:.4%}, CVaR: {cvar_99_full:.4%}")


Full history 95% VaR: 1.6590%, CVaR: 2.6996%
Full history 99% VaR: 3.2080%, CVaR: 4.7810%


In [18]:
var_95_250, cvar_95_250 = historical_var_cvar(returns_df_250, weights, CONFIDENCE_LEVELS[0])
var_99_250, cvar_99_250 = historical_var_cvar(returns_df_250, weights, CONFIDENCE_LEVELS[1])

print(f"{ROLLING_WINDOW_DAYS}-day {CONFIDENCE_LEVELS[0]:.0%} VaR: {var_95_250:.4%}, CVaR: {cvar_95_250:.4%}")
print(f"{ROLLING_WINDOW_DAYS}-day {CONFIDENCE_LEVELS[1]:.0%} VaR: {var_99_250:.4%}, CVaR: {cvar_99_250:.4%}")


250-day 95% VaR: 1.2188%, CVaR: 1.4941%
250-day 99% VaR: 1.5353%, CVaR: 1.8187%


Sanity checks to look at once these run:
- 99% VaR/CVaR should be larger (more negative loss) than 95% VaR/CVaR at the same window, since it looks further into the tail.
- CVaR should always be >= VaR (the tail average is at least as bad as the cutoff itself).
- Compare full-history vs 250-day: the 250-day window reflects only recent (likely calmer) conditions, so it will typically be smaller unless the last 250 days include a stress event.

## Parametric (Variance-Covariance / Delta-Normal) VaR / CVaR

Same story here — `parametric_var_cvar` is imported, not redefined locally.

In [19]:
var_95_param, cvar_95_param = parametric_var_cvar(returns_df, weights, CONFIDENCE_LEVELS[0])
var_99_param, cvar_99_param = parametric_var_cvar(returns_df, weights, CONFIDENCE_LEVELS[1])

print(f"Parametric (full history) {CONFIDENCE_LEVELS[0]:.0%} VaR: {var_95_param:.4%}, CVaR: {cvar_95_param:.4%}")
print(f"Parametric (full history) {CONFIDENCE_LEVELS[1]:.0%} VaR: {var_99_param:.4%}, CVaR: {cvar_99_param:.4%}")


Parametric (full history) 95% VaR: 1.8286%, CVaR: 2.3086%
Parametric (full history) 99% VaR: 2.6115%, CVaR: 3.0008%


In [20]:
var_95_param250, cvar_95_param250 = parametric_var_cvar(returns_df_250, weights, CONFIDENCE_LEVELS[0])
var_99_param250, cvar_99_param250 = parametric_var_cvar(returns_df_250, weights, CONFIDENCE_LEVELS[1])

print(f"Parametric ({ROLLING_WINDOW_DAYS}-day) {CONFIDENCE_LEVELS[0]:.0%} VaR: {var_95_param250:.4%}, CVaR: {cvar_95_param250:.4%}")
print(f"Parametric ({ROLLING_WINDOW_DAYS}-day) {CONFIDENCE_LEVELS[1]:.0%} VaR: {var_99_param250:.4%}, CVaR: {cvar_99_param250:.4%}")


Parametric (250-day) 95% VaR: 1.0974%, CVaR: 1.4059%
Parametric (250-day) 99% VaR: 1.6005%, CVaR: 1.8506%


In [21]:
as_of_date = returns_df.index.max()
portfolio_id = 1

rows = [
    {"date_id": as_of_date, "portfolio_id": portfolio_id, "method": "historical",
     "confidence": CONFIDENCE_LEVELS[0], "var_pct": var_95_250, "cvar_pct": cvar_95_250, "window_days": ROLLING_WINDOW_DAYS},
    {"date_id": as_of_date, "portfolio_id": portfolio_id, "method": "historical",
     "confidence": CONFIDENCE_LEVELS[1], "var_pct": var_99_250, "cvar_pct": cvar_99_250, "window_days": ROLLING_WINDOW_DAYS},
    {"date_id": as_of_date, "portfolio_id": portfolio_id, "method": "parametric",
     "confidence": CONFIDENCE_LEVELS[0], "var_pct": var_95_param250, "cvar_pct": cvar_95_param250, "window_days": ROLLING_WINDOW_DAYS},
    {"date_id": as_of_date, "portfolio_id": portfolio_id, "method": "parametric",
     "confidence": CONFIDENCE_LEVELS[1], "var_pct": var_99_param250, "cvar_pct": cvar_99_param250, "window_days": ROLLING_WINDOW_DAYS},
]

risk_metrics_df = pd.DataFrame(rows)
risk_metrics_df


,date_id,portfolio_id,method,confidence,var_pct,cvar_pct,window_days
0,2026-08-14,1,historical,0.95,0.012188,0.014941,250
1,2026-08-14,1,historical,0.99,0.015353,0.018187,250
2,2026-08-14,1,parametric,0.95,0.010974,0.014059,250
3,2026-08-14,1,parametric,0.99,0.016005,0.018506,250


In [22]:
insert_query = text("""
    INSERT INTO fact_risk_metrics (date_id, portfolio_id, method, confidence, var_pct, cvar_pct, window_days)
    VALUES (:date_id, :portfolio_id, :method, :confidence, :var_pct, :cvar_pct, :window_days)
    ON CONFLICT (date_id, portfolio_id, method, confidence)
    DO UPDATE SET var_pct = EXCLUDED.var_pct, cvar_pct = EXCLUDED.cvar_pct,
                  window_days = EXCLUDED.window_days, computed_at = now();
""")

with engine.begin() as conn:
    conn.execute(insert_query, risk_metrics_df.to_dict(orient="records"))